# 第5周 · 分类、评估与交叉验证

**本周结束时你能做到**：训练第一个分类器并画出它的决策边界；在不平衡数据上选对评估指标；用分层交叉验证可靠地比较逻辑回归与随机森林；识别并避免四类数据泄漏。

上周我们预测的是连续数值（回归）；这周开始预测类别（分类）：这位乘客生还了吗？这条影评是好评吗？这张图片是哪个数字？

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from zero2researcher import data
from zero2researcher.checks import week05 as ck

---
## S1 · 逻辑回归：把线性输出变成概率

线性回归输出任意实数，但"是不是好评"需要一个 0 到 1 之间的概率。逻辑回归在线性输出 $z = w \cdot x + b$ 外面套了一个 **sigmoid** 函数：

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

它把任意实数压进 (0, 1)：$z$ 很大时输出接近 1，$z$ 很小时接近 0，$z = 0$ 时恰好 0.5。

In [ ]:
def sigmoid(z):
    # 在这里写你的代码
    raise NotImplementedError

z = np.linspace(-8, 8, 200)
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(z, sigmoid(z))
ax.axhline(0.5, color="gray", linestyle=":")
ax.set_xlabel("Linear output z")
ax.set_ylabel("sigmoid(z)")
ax.set_title("Sigmoid squashes any number into (0, 1)")
plt.show()

### 2D 数据上的决策边界

造一份二维的两类数据，训练逻辑回归，然后对平面上每个点问模型"你觉得这是哪一类"——涂色出来就是**决策边界**。

In [ ]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression

X2, y2 = make_classification(n_samples=300, n_features=2, n_informative=2, n_redundant=0,
                             n_clusters_per_class=1, class_sep=1.2, random_state=42)
clf = LogisticRegression().fit(X2, y2)

grid_x, grid_y = np.meshgrid(np.linspace(X2[:, 0].min() - 1, X2[:, 0].max() + 1, 200),
                             np.linspace(X2[:, 1].min() - 1, X2[:, 1].max() + 1, 200))
proba = clf.predict_proba(np.c_[grid_x.ravel(), grid_y.ravel()])[:, 1].reshape(grid_x.shape)

fig, ax = plt.subplots(figsize=(5.5, 4.5))
cs = ax.contourf(grid_x, grid_y, proba, levels=20, cmap="RdBu_r", alpha=0.6)
fig.colorbar(cs, label="P(class 1)")
ax.scatter(X2[:, 0], X2[:, 1], c=y2, cmap="RdBu_r", edgecolor="k", s=18)
ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
ax.set_title("Logistic regression draws a straight decision boundary")
plt.show()

### 交叉熵：对正确答案感到意外的程度

逻辑回归的损失是**交叉熵**：模型给正确类别的概率是 $p$，损失就是 $-\log(p)$。

- 给正确答案 0.99 的概率 → 损失几乎为 0（毫不意外）。
- 给正确答案 0.01 的概率 → 损失巨大（"信誓旦旦地答错"被罚得最狠）。

训练就是不断调整 $w, b$，让模型在全部训练样本上的平均"意外程度"最小。

In [ ]:
p = np.linspace(0.001, 0.999, 300)
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(p, -np.log(p))
ax.set_xlabel("Predicted probability for the true class")
ax.set_ylabel("Loss  -log(p)")
ax.set_title("Confidently wrong predictions are punished hardest")
plt.show()
print("预测 0.9 时的损失:", round(-np.log(0.9), 3), "   预测 0.1 时的损失:", round(-np.log(0.1), 3))

### 混淆矩阵与四个指标

把预测和真相做成 2×2 的表，就是**混淆矩阵**（以"1 = 阳性"为约定）：

|  | 预测 1 | 预测 0 |
| --- | --- | --- |
| **真实 1** | TP（真阳性） | FN（假阴性，漏报） |
| **真实 0** | FP（假阳性，误报） | TN（真阴性） |

> 注意：下一格 sklearn 画的混淆矩阵按标签 0、1 的顺序排列——**左上角是 TN、右下角才是 TP**，行列顺序与上表正好相反。读任何混淆矩阵之前，先看清两条坐标轴各是什么。

- **准确率** accuracy = (TP + TN) / 全部 —— 预测对的比例
- **精确率** precision = TP / (TP + FP) —— 报出来的阳性里有多少是真的
- **召回率** recall = TP / (TP + FN) —— 真的阳性里抓住了多少
- **F1** = 精确率与召回率的调和平均；**macro-F1** 是对每一类分别算 F1 再取平均
- **ROC-AUC**：随机取一个正样本和一个负样本，模型给正样本更高分的概率

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, roc_auc_score

X2_tr, X2_te, y2_tr, y2_te = train_test_split(X2, y2, random_state=42, stratify=y2)
clf = LogisticRegression().fit(X2_tr, y2_tr)
pred = clf.predict(X2_te)

ConfusionMatrixDisplay.from_predictions(y2_te, pred)
plt.title("Confusion matrix on the held-out test set")
plt.show()
print(classification_report(y2_te, pred, digits=3))
print("ROC-AUC:", round(roc_auc_score(y2_te, clf.predict_proba(X2_te)[:, 1]), 3))

### 准确率什么时候骗人：不平衡数据

现实中的类别经常严重不平衡：罕见病筛查里 95% 的人是健康的。造一份 95:5 的数据看看——一个**什么都不学、永远喊"健康"** 的"模型"，准确率是多少？

（注意 `flip_y=0`：`make_classification` 默认会随机翻转 1% 的标签，会让比例偏离 95:5。）

In [ ]:
Xi, yi = make_classification(n_samples=2000, n_features=2, n_informative=2, n_redundant=0,
                             n_clusters_per_class=1, weights=[0.95, 0.05], flip_y=0,
                             class_sep=1.0, random_state=42)
print("两类的样本数:", np.bincount(yi))

# 在这里写你的代码
raise NotImplementedError
print("什么都不学、全预测多数类，准确率 =", majority_acc)
ck.imbalanced_majority_acc(majority_acc)

In [ ]:
Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(Xi, yi, random_state=42, stratify=yi)
clf_i = LogisticRegression().fit(Xi_tr, yi_tr)
print(classification_report(yi_te, clf_i.predict(Xi_te), digits=3))
print("ROC-AUC:", round(roc_auc_score(yi_te, clf_i.predict_proba(Xi_te)[:, 1]), 3))

对照上一格的输出读三个数：

- **准确率 0.950**——和那个什么都不学、永远喊"健康"的基线一模一样；
- 少数类（`1`）的**召回率只有 0.28**：测试集里 25 个真正的阳性只抓到 7 个，漏掉了七成多（多数类的召回率是 0.985）；
- **macro-F1 只有约 0.67**，一眼就暴露了问题。

ROC-AUC 却有约 0.97：模型给阳性打的分整体上确实更高（排序能力不差），坏就坏在默认的 0.5 阈值上。ROC-AUC 不管阈值选得好不好，所以它要和混淆矩阵、macro-F1 一起看。

**结论**：不平衡数据不要只看准确率；看混淆矩阵、macro-F1（它对每一类一视同仁），以及与阈值无关的 ROC-AUC。

---
## S2 · 为什么需要验证集与交叉验证

在训练集上评估自己，就像出题人给自己打分；用测试集反复调参，就像考试前偷看了答案——两者的分数都不可信。标准做法：

- **训练集**：学参数。**验证集**：选模型、调超参数。**测试集**：最后只用一次，报告成绩。
- 数据不多时用 **k 折交叉验证**：切成 k 份，轮流用 k−1 份训练、1 份验证，报告 k 个分数的均值 ± 标准差。
- 分类任务用 **StratifiedKFold**（分层 k 折）：每一折里两类的比例都和整体一致——不平衡数据尤其重要。

### 实战：Titanic 生存预测

用四个特征预测 `survived`：`sex`、`pclass`、`age`（有 177 个缺失）、`fare`。

In [ ]:
titanic = data.load_titanic()
X = titanic[["pclass", "age", "fare"]].copy()
X["sex"] = (titanic["sex"] == "female").astype(int)   # female=1, male=0
y = titanic["survived"]
print(X.isna().sum())
X.head()

### 预处理必须放进 Pipeline

"用中位数填充缺失"与"标准化"都是**从数据里学出来的步骤**（中位数、均值、标准差都是参数！）。验证折和测试集扮演的是"未来的数据"——真实使用时，你不可能提前拿到它们来算中位数。所以如果先在全部数据上算好、再做交叉验证，验证折的信息就提前泄漏给了训练。Kaggle 的数据泄漏教程把话说得很直白：*"exclude the validation data from any type of fitting, including the fitting of preprocessing steps"*；scikit-learn 文档的 Common pitfalls 一节给出的总规则也是：永远不要在测试数据上调用 `fit`。

做法：把 `SimpleImputer`（中位数填充）→ `StandardScaler` → `LogisticRegression` 打包成一个 **Pipeline**，再把整个 Pipeline 交给交叉验证——每一折内部自动只用训练折去 fit 预处理。**这是本课程从今天起的标准写法。**

自查函数会同时检查你交给交叉验证的 Pipeline 和 `X`：填充与标准化必须在 Pipeline 里，`X` 必须是还带着缺失值的原始特征。

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate

# 在这里写你的代码
raise NotImplementedError
print("5 折准确率:", np.round(scores, 3))
print(f"平均 {scores.mean():.3f} ± {scores.std():.3f}")
ck.titanic_cv_scores(scores, lr_pipe, X)

约 0.79 ± 0.01。注意**标准差和均值一样重要**：两个模型差 0.5 个百分点、而折间波动有 3 个百分点时，这个差距什么都说明不了。

### 学习曲线：更多数据有用吗？

把训练集从小到大，画"训练分数 vs 验证分数"两条曲线：

In [ ]:
from sklearn.model_selection import learning_curve

sizes, tr_sc, va_sc = learning_curve(lr_pipe, X, y, cv=cv,
                                     train_sizes=np.linspace(0.1, 1.0, 6),
                                     shuffle=True, random_state=42)
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(sizes, tr_sc.mean(axis=1), "o-", label="train")
ax.plot(sizes, va_sc.mean(axis=1), "o-", label="validation")
ax.set_xlabel("Training set size")
ax.set_ylabel("Accuracy")
ax.set_title("Both curves plateau: more data alone won't help much")
ax.legend()
plt.show()

读图口诀：
- 两条线都低、且已贴在一起 → **欠拟合**，换更强的模型或更好的特征，加数据没用。
- 训练线高、验证线低、缺口大 → **过拟合**，加数据或加正则化有望缩小缺口。

### 验证曲线：正则化强度 C

逻辑回归用参数 `C` 控制正则化强度：**C 越小，正则化越强**（模型被压得越"平"）。

> ⚠️ sklearn **1.8** 起 `LogisticRegression` 的 `penalty` 参数已弃用（1.10 将移除）。旧教程里的 `penalty="l2"` 写法不要再用：正则化的**强度**用 `C` 调，正则化的**类型**用 `l1_ratio` 选——`l1_ratio=0`（默认）是 L2，`l1_ratio=1` 是 L1，介于两者之间是弹性网。默认的 `lbfgs` 求解器只支持 L2；要用 L1 得换 `solver="liblinear"` 或 `"saga"`，弹性网只有 `"saga"` 支持。
>
> L1 和 L2 的直观区别：**L2** 把所有权重整体往 0 收缩，但很少恰好变成 0；**L1** 会把不重要特征的权重直接压成 0，相当于顺带做了特征选择——在基因表达这类"特征多、样本少"的数据上很常用。本课程默认用 L2。

In [ ]:
Cs = [0.001, 0.01, 0.1, 1, 10, 100]
tr_means, va_means = [], []
for C in Cs:
    pipe = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                         LogisticRegression(C=C, max_iter=1000))
    res = cross_validate(pipe, X, y, cv=cv, return_train_score=True)
    tr_means.append(res["train_score"].mean())
    va_means.append(res["test_score"].mean())

fig, ax = plt.subplots(figsize=(5, 3))
ax.semilogx(Cs, tr_means, "o-", label="train")
ax.semilogx(Cs, va_means, "o-", label="validation")
ax.set_xlabel("C (inverse regularization strength)")
ax.set_ylabel("Accuracy")
ax.set_title("Too much regularization (small C) underfits")
ax.legend()
plt.show()

---
## S3 · 决策树与随机森林

决策树就是"20 个问题"游戏：每个节点问一个是/否问题（"是女性吗？""票价 > 10 吗？"），一路问到叶子，叶子里多数人的命运就是预测。先看一棵浅树长什么样：

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

tree_pipe = make_pipeline(SimpleImputer(strategy="median"),
                          DecisionTreeClassifier(max_depth=3, random_state=42))
tree_pipe.fit(X, y)

fig, ax = plt.subplots(figsize=(11, 5))
plot_tree(tree_pipe[-1], feature_names=X.columns, class_names=["died", "survived"],
          filled=True, fontsize=8, ax=ax)
ax.set_title("A depth-3 decision tree is a flowchart of yes/no questions")
plt.show()

第一个问题就是性别——树自己从数据里学出了"妇女优先"。

单棵深树很容易过拟合（把训练集背下来）。**随机森林**：训练很多棵"各自见过不同的 bootstrap 样本（有放回抽样得到的数据子集），并且每次分裂只在随机抽出的几个特征里挑"的树，预测时投票——个体的过拟合在平均中互相抵消。这叫**集成**（ensemble）。三个最常用的超参数：

- `n_estimators`：树的数量（默认 100，多一般更稳但更慢）
- `max_depth`：每棵树的最大深度（默认不限制）
- `max_features`：每次分裂随机考虑的特征数（默认 `sqrt`，这是"随机"的来源之一）

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# 在这里写你的代码
raise NotImplementedError
for name, pipe in [("LogisticRegression", lr_pipe), ("RandomForest", rf_pipe)]:
    s = cross_val_score(pipe, X, y, cv=cv)
    print(f"{name:20s} {s.mean():.3f} ± {s.std():.3f}")

同一套交叉验证下随机森林约 0.83，比逻辑回归高 4 个百分点左右——在同一个 `cv` 上比较才公平。

### 特征重要性及其陷阱

随机森林自带 `feature_importances_`（基于不纯度下降），但它**偏爱取值多的连续特征**（age、fare 有几百种取值，sex 只有 2 种，连续特征有更多机会被选来分裂）。更可信的是 **permutation importance**：在留出的数据上把某一列打乱，看分数掉多少。

In [ ]:
from sklearn.inspection import permutation_importance

X_tr, X_te, y_tr, y_te = train_test_split(X, y, random_state=42, stratify=y)
rf_pipe.fit(X_tr, y_tr)
imp = rf_pipe[-1].feature_importances_
perm = permutation_importance(rf_pipe, X_te, y_te, n_repeats=10, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].bar(X.columns, imp)
axes[0].set_title("Impurity importance (biased)")
axes[1].bar(X.columns, perm.importances_mean)
axes[1].set_title("Permutation importance (held-out data)")
for a in axes:
    a.set_ylabel("Importance")
plt.tight_layout()
plt.show()

左图说 fare 最重要；右图（打乱后掉分最多的）说 **sex** 才是——与我们对这场灾难的了解一致。读论文看到"特征重要性"时，先问一句：是哪一种？

---
## S4 · 数据泄漏：好得不像话的结果，多半是假的

**数据泄漏**＝评估时用到了"真实预测场景下不可能有"的信息。四种常见形态：

1. **目标泄漏**：特征里混进了"答案之后才会产生"的信息。例：用"是否服用了抗生素"预测"是否患肺炎"——服药发生在确诊之后。
2. **重复样本跨集合**：同一条样本既在训练集又在测试集（数据集拼接、去重不彻底时非常常见）。
3. **在全部数据上先做预处理/特征选择，再划分**：标准化、填充、选特征都"看过"测试集。
4. **同组样本跨集合**：同一位病人的多条记录、同一次实验的多个重复，被拆进了训练集和测试集。做过实验的人秒懂：这相当于把**技术重复**当成了**生物学重复**。解法是按组划分（`GroupKFold`，新版本也支持 `shuffle=True`）。

第 4 条在生物数据里最常见，也最容易被忽视。先看它在代码里长什么样：40 位病人，每人 10 条记录（技术重复）；每位病人的读数有自己的"指纹"，而标签（患病与否）与读数**毫无关系**——任何模型的真实准确率都只能约 0.5。

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier

rng = np.random.default_rng(42)
pat_label = rng.permutation(np.array([0, 1] * 20))           # 40 位病人，各自一个标签，与特征无真实关系
pat_profile = rng.normal(0, 1, size=(40, 5))                 # 每位病人自己的"指纹"
groups = np.repeat(np.arange(40), 10)                        # 每人 10 条记录（技术重复）
X_g = pat_profile[groups] + rng.normal(0, 0.3, size=(400, 5))
y_g = pat_label[groups]

rf_g = RandomForestClassifier(n_estimators=100, random_state=42)
acc_by_record = cross_val_score(rf_g, X_g, y_g, cv=StratifiedKFold(5, shuffle=True, random_state=42)).mean()
acc_by_patient = cross_val_score(rf_g, X_g, y_g, cv=GroupKFold(5, shuffle=True, random_state=42),
                                 groups=groups).mean()
print(f"按记录拆（StratifiedKFold）：{acc_by_record:.3f}")
print(f"按病人拆（GroupKFold）：    {acc_by_patient:.3f}   ← 标签与特征无关，真实水平只能约 0.5")

按记录拆时，验证折里每条记录的"兄弟记录"都在训练折里，模型只要认出"这是哪位病人"就能答对——它学到的是病人的指纹，不是疾病。按病人拆（`GroupKFold`，把 `groups=` 传给 `cross_val_score`）以后，验证折里全是没见过的病人，准确率立刻回到抛硬币的水平。**有效样本量是 40 位病人，不是 400 条记录。**

只要数据里有"组"（同一病人、同一供体、同一批次、同一只小鼠、同一天的实验），就要问一句：部署时遇到的是新的组吗？是的话，就按组划分。

### 现场重演：ESL §7.10.2 的"错误的交叉验证"

《The Elements of Statistical Learning》7.10.2 节的经典实验：50 个样本、两类各半，5000 个**与标签完全无关**的标准高斯特征。任何模型的真实误差只能是 50%。现在用一个"看起来很合理"的流程：

1. 在**全部 50 个样本**上，选出与标签相关性最强的 100 个特征；
2. 只用这 100 个特征建一个 1-最近邻分类器；
3. 交叉验证估计误差。

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

rng = np.random.default_rng(42)
N, P, K = 50, 5000, 100
X_rand = rng.standard_normal((N, P))
y_rand = rng.permutation(np.array([0] * 25 + [1] * 25))


def top_corr_features(Xs, ys, k=100):
    """返回与标签相关系数绝对值最大的 k 个特征的列号"""
    Xc = Xs - Xs.mean(axis=0)
    yc = ys - ys.mean()
    corr = (Xc * yc[:, None]).sum(axis=0) / (
        np.sqrt((Xc ** 2).sum(axis=0)) * np.sqrt((yc ** 2).sum()) + 1e-12)
    return np.argsort(-np.abs(corr))[:k]

In [ ]:
# 错误流程：先在全部样本上选特征，再交叉验证
# 在这里写你的代码
raise NotImplementedError
print(f"错误流程的 CV 误差：{wrong_err:.1%}   （特征和标签明明毫无关系！）")
ck.wrong_cv_error(wrong_err)

In [ ]:
# 正确流程：特征选择放进每一折内部
# 在这里写你的代码
raise NotImplementedError
print(f"正确流程的 CV 误差：{right_err:.1%}   —— 和抛硬币一样，这才是真相")
ck.right_cv_error(right_err)

**为什么会这样？** 5000 个纯随机特征里，总有一些**碰巧**和这 50 个标签相关。在全部样本上挑出的这 100 个"幸运儿"，在留出折上也保留着这份巧合（ESL 的原文：它们与留出样本标签的相关系数平均约 0.28，而不是 0）。书中 50 次模拟的平均 CV 误差只有 **3%**——真实误差是 **50%**。

**规则（本课程的底线）**：**一切从数据里学参数的步骤**——填充用的中位数、标准化用的均值和标准差、TF-IDF 的词表和 idf（第 6 周会学）、特征选择——都放进 Pipeline，只在训练折（或训练集）上 fit。理由就是 S2 那句话：测试集代表"未来的数据"，对它的任何统计，真实使用时都不可能提前拿到。Kaggle 的泄漏教程和 scikit-learn 文档给的都是这条规则。

**它和 ESL 的说法并不矛盾。** ESL 原文的限定是：**不看标签**的无监督筛选（比如按方差筛基因）可以在划分之前做——这是统计意义上的判断：不用标签的步骤没法把"碰巧和标签相关"的巧合带进来，所以不会像上面那样把误差从 50% 压到 2%。两者的差别在于后果大小：

| 场景 | 这一步放在划分前 | 放在 CV 的每一折内部 |
| --- | --- | --- |
| Titanic：中位数填充 + 标准化（不看标签），逻辑回归 5 折准确率 | 0.789 | 0.788 |
| ESL：按与标签的相关性选特征（用到标签），1-NN 的 CV 误差 | 0.02 | 0.50 |

不看标签的步骤放错位置，在本课程的数据上影响很小（不到 0.2 个百分点）；用到标签的步骤放错位置，结论会彻底颠倒。但"很小"不等于"没有"，而且你事先无法确定它在自己的数据上有多大——所以本课程一律按底线做。记成两句话：

1. **所有 fit 都只用训练数据**（底线，全部装进 Pipeline）；
2. **用到标签的步骤放错，后果最严重。**

### 本周小结

- 逻辑回归 = 线性模型 + sigmoid；损失是交叉熵（"对正确答案的意外程度"）
- 混淆矩阵、精确率/召回率/F1/ROC-AUC；不平衡数据别只看准确率
- train/val/test；StratifiedKFold 交叉验证；报告均值 ± 标准差
- 学习曲线判断"要不要更多数据"；验证曲线调 `C`（sklearn 1.8 起别再写 `penalty`，L1 / L2 用 `l1_ratio` 选）
- 决策树是提问流程图；随机森林靠"集体投票"抗过拟合；特征重要性要用 permutation 版
- 四类数据泄漏；同组样本要按组划分（`GroupKFold`）；**一切 fit（填充、标准化、选特征……）都只用训练折，全部装进 Pipeline**；用到标签的步骤放错，后果最严重